# 🔬 OSFT 파인튜닝

## OSFT (Orthogonal Subspace Fine-Tuning)란?

OSFT는 가중치 행렬을 직교 분해하여 선택적으로 학습하는 파인튜닝 기법입니다.

### LoRA와의 차이점

| 구분 | LoRA | OSFT |
|------|------|------|
| **접근 방식** | 저차원 어댑터 행렬 추가 | 가중치 직교 분해 후 선택적 해동 |
| **메모리** | 어댑터만큼 추가 메모리 | 분해 + 활성화 + 옵티마이저 메모리 |
| **파라미터** | rank, alpha | unfreeze_rank_ratio |
| **모듈성** | 어댑터 분리 가능 | 전체 모델에 반영됨 |
| **망각 저항** | 원본 보존으로 양호 | 직교 부분 공간으로 간섭 최소화 |

### OSFT의 특징
- 가중치 행렬 W를 직교 분해: W = U·Σ·V^T
- `unfreeze_rank_ratio`만큼의 특이값/벡터만 학습 대상으로 선택
- 나머지 직교 부분 공간은 고정 → 기존 지식 보존에 유리
- LoRA보다 더 많은 메모리를 사용하지만, 망각 저항이 우수할 수 있음

### 이 노트북의 설정
- **기본 모델**: `Qwen/Qwen3-4B-Instruct-2507` (LoRA와 동일)
- **unfreeze_rank_ratio**: 0.25 (전체 rank의 25% 학습)
- **데이터**: LoRA와 동일한 정규 데이터 (독립 학습)

> ⚠️ OSFT는 LoRA와 **독립적**으로 실행됩니다.  
> 동일한 기본 모델, 동일한 학습 데이터에서 시작합니다.  
> LoRA 체크포인트 위에 OSFT를 적용하지 **않습니다**.

In [ ]:
"""환경 확인 — 00_preflight.ipynb 에서 이미 설치 완료."""

import os
from pathlib import Path

_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

os.environ["RHOAI_PROJECT_ROOT"] = str(_project_root)

try:
    import rhoai_model_training_lab.config as _cfg
    _cfg.PROJECT_ROOT = _project_root
    print(f"✅ 프로젝트 루트: {_project_root}")
except ImportError:
    raise ImportError(
        "❌ 패키지 미설치 — 먼저 00_preflight.ipynb 를 실행하세요."
    )


In [ ]:
"""Load OSFT config and validate bundle compatibility."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_training_config, load_bundle_config, PROJECT_ROOT,
)
from rhoai_model_training_lab.data import BundleManager

load_env()

# Load OSFT config
osft_config = load_training_config("osft")
model_id = osft_config["model"]["model_id"]
model_revision = osft_config["model"]["model_revision"]

print(f"모델: {model_id} (rev: {model_revision})")
print(f"OSFT unfreeze_rank_ratio: {osft_config['osft']['unfreeze_rank_ratio']}")
print(f"시드: {osft_config['training']['seed']}")

# Load and validate bundle (same bundle as LoRA)
release_config = load_bundle_config()
bundle_base = release_config.get("bundle", {}).get("base_path", "data/prepared/tau-knowledge-v1")
bundle_path = PROJECT_ROOT / bundle_base

print(f"\n번들 경로: {bundle_path}")
mgr = BundleManager.load_bundle(bundle_path)
manifest = mgr.manifest

# Compatibility check (same base model as LoRA)
compat = mgr.validate_compatibility(model_id)
if compat.errors:
    for err in compat.errors:
        print(f"  ❌ {err}")
    raise RuntimeError(
        "번들 호환성 검증 실패 — 학습을 진행할 수 없습니다.\n"
        "data_preparation/ 노트북에서 올바른 번들을 생성하세요."
    )

print(f"\n✅ 번들 호환성 검증 통과")
print(f"   학습 샘플: {manifest.canonical_train_count} (LoRA와 동일)")
print(f"   검증 샘플: {manifest.canonical_validation_count}")
print(f"   번들 버전: {manifest.bundle_version}")

In [ ]:
"""Preview training data with OSFT-specific formatting."""

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Load OSFT-specific training data
train_data = mgr.get_training_samples("osft", "train")
val_data = mgr.get_training_samples("osft", "validation")

print(f"OSFT 학습 데이터: {len(train_data)} 샘플")
print(f"OSFT 검증 데이터: {len(val_data)} 샘플")

# Verify sample parity with LoRA
lora_train = mgr.get_training_samples("lora", "train")
print(f"\n📊 LoRA 학습 샘플 수: {len(lora_train)}")
print(f"   OSFT 학습 샘플 수: {len(train_data)}")
if len(train_data) == len(lora_train):
    print("   ✅ 샘플 수 일치 — 동일한 정규 데이터에서 변환")
else:
    print("   ⚠️  샘플 수 불일치 — 백엔드별 변환 차이 확인 필요")

# Preview with OSFT formatting focus
print("\n" + "=" * 70)
print("📝 OSFT 학습 데이터 미리보기")
print("=" * 70)

for i, sample in enumerate(train_data[:2]):
    messages = sample.get("messages", [])
    print(f"\n--- 샘플 {i+1} ---")
    print(f"메시지 수: {len(messages)}")

    for msg in messages:
        role = msg["role"]
        content = msg.get("content", "") or ""
        is_target = role == "assistant"
        mask_icon = "🎯 [학습 대상]" if is_target else "🚫 [마스킹됨]"

        display = content[:200] + "..." if len(content) > 200 else content
        print(f"\n  {mask_icon} [{role}]: {display}")

    # Token analysis
    try:
        tokens = tokenizer.apply_chat_template(messages, tokenize=True)
        print(f"\n  토큰 수: {len(tokens)}")
    except Exception:
        pass

# Memory estimation note
print(f"\n{'=' * 70}")
print("💾 OSFT 메모리 참고사항:")
print(f"  - unfreeze_rank_ratio: {osft_config['osft']['unfreeze_rank_ratio']}")
print("  - OSFT는 LoRA보다 더 많은 GPU 메모리를 사용합니다")
print("  - 분해(decomposition) + 활성화(activation) + 옵티마이저 상태 포함")
print(f"  - 배치 크기: {osft_config['training_args']['per_device_train_batch_size']}")
print(f"  - 기울기 누적: {osft_config['training_args']['gradient_accumulation_steps']}")

In [ ]:
"""training_hub.osft 로 OSFT 학습 실행."""

import time
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 감지되지 않았습니다. OSFT 학습에는 GPU가 필요합니다.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
print()

train_file = str(PROJECT_ROOT / osft_config["data"]["train_file"])
val_file = str(PROJECT_ROOT / osft_config["data"]["validation_file"])
output_dir = str(PROJECT_ROOT / osft_config["training_args"]["output_dir"])

print("=" * 70)
print("🚀 OSFT 학습 시작")
print("=" * 70)
print(f"  모델: {model_id}")
print(f"  학습 데이터: {train_file}")
print(f"  검증 데이터: {val_file}")
print(f"  출력: {output_dir}")
print(f"  에포크: {osft_config['training_args']['num_train_epochs']}")
print(f"  배치: {osft_config['training_args']['per_device_train_batch_size']} x {osft_config['training_args']['gradient_accumulation_steps']}")
print(f"  학습률: {osft_config['training_args']['learning_rate']}")


mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
mlflow_experiment = os.environ.get("MLFLOW_EXPERIMENT_TRAINING", "rhoai-model-training-lab-training")

start_time = time.time()

# Workaround: training_hub passes 'logging_dir' to SFTConfig
# but TRL 1.x + Unsloth removed this parameter
from trl import SFTConfig as _SFTConfig
_orig_sft_init = _SFTConfig.__init__
def _compat_sft_init(self, *args, logging_dir=None, **kwargs):
    _orig_sft_init(self, *args, **kwargs)
_SFTConfig.__init__ = _compat_sft_init

from training_hub import osft as osft_train

training_result = osft_train(
    model_path=model_id,
    data_path=train_file,
    ckpt_output_dir=output_dir,

    # Training
    num_epochs=osft_config["training_args"]["num_train_epochs"],
    micro_batch_size=osft_config["training_args"]["per_device_train_batch_size"],
    gradient_accumulation_steps=osft_config["training_args"]["gradient_accumulation_steps"],
    learning_rate=osft_config["training_args"]["learning_rate"],
    max_seq_len=osft_config["data"]["max_seq_length"],
    lr_scheduler=osft_config["training_args"]["lr_scheduler_type"],
    bf16=osft_config["training_args"]["bf16"],
    flash_attention=True,
    trust_remote_code=True,

    # Logging & Saving
    logging_steps=osft_config["training_args"]["logging_steps"],
    eval_steps=osft_config["training_args"]["eval_steps"],
    save_steps=osft_config["training_args"]["save_steps"],
    save_total_limit=osft_config["training_args"]["save_total_limit"],

    # Evaluation
    eval_data_path=val_file,

    # MLflow
    mlflow_tracking_uri=mlflow_uri or None,
    mlflow_experiment_name=mlflow_experiment if mlflow_uri else None,
    mlflow_run_name=f"osft-{manifest.bundle_version}" if mlflow_uri else None,

    # Dataset format
    dataset_type="chat",
    field_messages="messages",
)

wall_time = time.time() - start_time

print(f"\n✅ OSFT 학습 완료!")
print(f"  소요 시간: {wall_time/60:.1f}분")
print(f"  피크 VRAM: {torch.cuda.max_memory_allocated() / (1024**3):.1f} GB")


In [ ]:
"""Training analysis — loss curves, learning rate, gradient norm."""

import json
from pathlib import Path

output_dir_path = Path(output_dir)

trainer = None
if isinstance(training_result, dict):
    trainer = training_result.get("trainer")

log_history = None
if trainer and hasattr(trainer, "state") and hasattr(trainer.state, "log_history"):
    log_history = trainer.state.log_history

if log_history is None:
    state_path = output_dir_path / "trainer_state.json"
    if state_path.exists():
        with open(state_path) as f:
            log_history = json.load(f).get("log_history", [])

if not log_history:
    print("⚠️  No training logs found.")
    train_losses, eval_losses, train_steps, eval_steps = [], [], [], []
else:
    train_steps = [e["step"] for e in log_history if "loss" in e]
    train_losses = [e["loss"] for e in log_history if "loss" in e]
    eval_steps = [e["step"] for e in log_history if "eval_loss" in e]
    eval_losses = [e["eval_loss"] for e in log_history if "eval_loss" in e]
    lr_steps = [e["step"] for e in log_history if "learning_rate" in e]
    lr_values = [e["learning_rate"] for e in log_history if "learning_rate" in e]
    grad_steps = [e["step"] for e in log_history if "grad_norm" in e]
    grad_norms = [e["grad_norm"] for e in log_history if "grad_norm" in e]

    try:
        import matplotlib.pyplot as plt
        import numpy as np

        n_plots = 2 + bool(lr_values) + bool(grad_norms)
        fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4.5))
        if n_plots == 1:
            axes = [axes]
        ax_idx = 0

        # 1. Loss curve
        ax = axes[ax_idx]; ax_idx += 1
        ax.plot(train_steps, train_losses, alpha=0.4, color="#4C72B0", linewidth=0.8, label="Train (raw)")
        if len(train_losses) > 5:
            smoothed = [train_losses[0]]
            for v in train_losses[1:]:
                smoothed.append(0.1 * v + 0.9 * smoothed[-1])
            ax.plot(train_steps, smoothed, color="#4C72B0", linewidth=2, label="Train (smoothed)")
        if eval_losses:
            ax.plot(eval_steps, eval_losses, color="#DD8452", marker="o", markersize=4,
                    linewidth=2, label="Eval loss")
        ax.set_xlabel("Step"); ax.set_ylabel("Loss"); ax.set_title("Loss Curve")
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

        # 2. Generalization gap
        ax = axes[ax_idx]; ax_idx += 1
        if eval_losses and len(eval_losses) > 1:
            ax.plot(eval_steps, eval_losses, "o-", color="#DD8452", label="Eval", markersize=5)
            interp_train = np.interp(eval_steps, train_steps, train_losses)
            ax.plot(eval_steps, interp_train, "s--", color="#4C72B0", label="Train (at eval)", markersize=4)
            gap = [e - t for e, t in zip(eval_losses, interp_train)]
            ax.fill_between(eval_steps, interp_train, eval_losses, alpha=0.15, color="red",
                           label=f"Gap (final: {gap[-1]:+.4f})")
            ax.set_title("Generalization Gap")
        else:
            ax.plot(train_steps, train_losses, color="#4C72B0")
            ax.set_title("Train Loss Detail")
        ax.set_xlabel("Step"); ax.set_ylabel("Loss")
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

        # 3. LR schedule
        if lr_values:
            ax = axes[ax_idx]; ax_idx += 1
            ax.plot(lr_steps, lr_values, color="#55A868", linewidth=1.5)
            ax.set_xlabel("Step"); ax.set_ylabel("Learning Rate"); ax.set_title("LR Schedule")
            ax.ticklabel_format(axis="y", style="scientific", scilimits=(-4, -4))
            ax.grid(True, alpha=0.3)

        # 4. Gradient norm
        if grad_norms:
            ax = axes[ax_idx]; ax_idx += 1
            ax.plot(grad_steps, grad_norms, color="#C44E52", alpha=0.6, linewidth=0.8)
            if len(grad_norms) > 5:
                sg = [grad_norms[0]]
                for v in grad_norms[1:]:
                    sg.append(0.1 * v + 0.9 * sg[-1])
                ax.plot(grad_steps, sg, color="#C44E52", linewidth=2, label="Smoothed")
                ax.legend(fontsize=8)
            ax.set_xlabel("Step"); ax.set_ylabel("Gradient Norm"); ax.set_title("Gradient Norm")
            ax.grid(True, alpha=0.3)

        fig.suptitle("OSFT Training Analysis", fontsize=13, fontweight="bold", y=1.02)
        plt.tight_layout()
        plt.show()

    except ImportError:
        pass

    print("\n" + "=" * 60)
    print("Training Metrics Summary")
    print("=" * 60)
    if train_losses:
        print(f"  Initial loss:  {train_losses[0]:.4f}")
        print(f"  Final loss:    {train_losses[-1]:.4f}")
        print(f"  Min loss:      {min(train_losses):.4f}")
        reduction = (1 - train_losses[-1] / train_losses[0]) * 100
        print(f"  Reduction:     {reduction:.1f}%")
    if eval_losses:
        print(f"  Final eval:    {eval_losses[-1]:.4f}")
        print(f"  Best eval:     {min(eval_losses):.4f}")
        best_step = eval_steps[eval_losses.index(min(eval_losses))]
        print(f"  Best eval at:  step {best_step}")
        if len(eval_losses) > 1 and eval_losses[-1] > min(eval_losses) * 1.05:
            print(f"  ⚠️  Possible overfitting (eval loss increased from best)")
    print(f"\n  Total steps:   {train_steps[-1] if train_steps else 'N/A'}")
    print(f"  Peak VRAM:     {torch.cuda.max_memory_allocated() / (1024**3):.1f} GB")
    print(f"  Wall time:     {wall_time/60:.1f} min")


In [ ]:
"""학습 결과 로컬 저장."""

result_summary = {
    "method": "osft",
    "model_id": model_id,
    "bundle_name": manifest.bundle_name,
    "bundle_version": manifest.bundle_version,
    "train_samples": manifest.canonical_train_count,
    "validation_samples": manifest.canonical_validation_count,
    "learning_rate": osft_config["training_args"]["learning_rate"],
    "num_epochs": osft_config["training_args"]["num_train_epochs"],
    "wall_time_seconds": wall_time,
    "peak_vram_gb": torch.cuda.max_memory_allocated() / (1024**3),
    "gpu_name": torch.cuda.get_device_name(0),
    "final_train_loss": train_losses[-1] if train_losses else None,
    "final_eval_loss": eval_losses[-1] if eval_losses else None,
}

result_path = Path(output_dir) / "training_result.json"
result_path.parent.mkdir(parents=True, exist_ok=True)
with open(result_path, "w") as f:
    json.dump(result_summary, f, indent=2, ensure_ascii=False)
print(f"📄 학습 결과 저장: {result_path}")

if mlflow_uri:
    print(f"📊 MLflow 기록: {mlflow_uri}")


In [ ]:
"""Verify OSFT checkpoint."""

from rich.table import Table
from rich.console import Console

console = Console()

checkpoint_dir = Path(output_dir)
checks = []

# Check for model files
has_model_files = (
    list(checkpoint_dir.glob("*.safetensors"))
    or list(checkpoint_dir.glob("*.bin"))
    or list(checkpoint_dir.glob("model*.safetensors"))
)
checks.append(("모델 가중치 파일", bool(has_model_files)))

# Check config
has_config = (checkpoint_dir / "config.json").exists()
checks.append(("모델 설정 파일", has_config))

# Check tokenizer
has_tokenizer = (checkpoint_dir / "tokenizer_config.json").exists()
checks.append(("토크나이저 파일", has_tokenizer))

# Try reloading
reload_ok = False
try:
    from transformers import AutoModelForCausalLM

    # Just verify the config can be loaded (don't load full model)
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(str(checkpoint_dir), trust_remote_code=True)
    reload_ok = True
    print(f"✅ 모델 설정 리로드 성공: {cfg.model_type}")
except Exception as exc:
    print(f"⚠️  모델 설정 리로드 실패: {exc}")

checks.append(("모델 리로드 검증", reload_ok))

# Checkpoint size
total_size = sum(f.stat().st_size for f in checkpoint_dir.rglob("*") if f.is_file())
size_gb = total_size / (1024**3)
checks.append((f"체크포인트 크기 ({size_gb:.2f} GB)", size_gb > 0))

# Summary table
table = Table(title="🔍 OSFT 체크포인트 검증", show_header=True)
table.add_column("항목", style="bold")
table.add_column("상태")

for name, ok in checks:
    status = "✅" if ok else "❌"
    table.add_row(name, status)

console.print(table)

all_ok = all(ok for _, ok in checks)
if all_ok:
    print("\n🎉 OSFT 학습이 성공적으로 완료되었습니다!")
    print("\n다음 단계:")
    print("  📓 05_export_and_deploy.ipynb — 모델 내보내기 및 배포")
    print("  LoRA와 OSFT 체크포인트가 모두 준비되었으므로 비교 평가가 가능합니다.")
else:
    print("\n⚠️  일부 검증 항목이 실패했습니다. 위 결과를 확인하세요.")